医院后续处理成医院-坐标数据的全过程：1、百度坐标和高德坐标系转换成WGS坐标，计算两套坐标系下坐标的距离。2、加入工商数据、手动查找数据，得到较为齐全的医院坐标数据


In [1]:
# 百度坐标&高德坐标转换成WGS84坐标 eviltransform方法

import pandas as pd
import eviltransform
import logging
from pathlib import Path
from enum import Enum

In [2]:
import pandas as pd
from pathlib import Path
import logging
from enum import Enum
import eviltransform

# ------------------------------
# 坐标系统枚举
# ------------------------------
class CoordSystem(Enum):
    AMAP = "amap"      # 高德 GCJ-02
    BAIDU = "baidu"    # 百度 BD-09

# ------------------------------
# 配置区
# ------------------------------
class Config:
    def __init__(self):
        self.base_dir = Path(
            r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API"
        )
        
        self.input_file = self.base_dir / "hospital_place_result.csv"
        self.output_file = self.base_dir / "hospital_place_baidu_gaode_wgs84.csv"
        
        self.columns_map = {
            CoordSystem.BAIDU: {
                "lng": "百度经度",
                "lat": "百度纬度",
                "out_lng": "百度_wgs84经度1",
                "out_lat": "百度_wgs84纬度1",
            },
            CoordSystem.AMAP: {
                "lng": "高德经度",
                "lat": "高德纬度",
                "out_lng": "高德_wgs84经度",
                "out_lat": "高德_wgs84纬度",
            },
        }
        
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s"
        )

config = Config()

# ------------------------------
# 加载文件（增强版本）
# ------------------------------
def load_file(path: Path) -> pd.DataFrame:
    # 尝试更全面的编码列表
    encodings = [
        "utf-8-sig", "utf-8", "gbk", "gb2312", "gb18030", 
        "ansi", "iso-8859-1", "latin1", "cp1252", "cp1250",
        "big5", "shift_jis", "euc-jp", "euc-kr"
    ]
    
    # 首先检查文件是否存在
    if not path.exists():
        raise FileNotFoundError(f"文件不存在：{path}")
    
    print(f"尝试加载文件：{path}")
    print(f"文件大小：{path.stat().st_size} 字节")
    
    # 尝试读取前几行来查看内容
    try:
        with open(path, 'rb') as f:
            sample = f.read(500)
            print(f"文件前500字节（十六进制）：{sample[:100].hex()}...")
            print(f"文件前500字节（文本尝试解码为utf-8）：{sample[:100].decode('utf-8', errors='ignore')}")
    except Exception as e:
        print(f"无法读取文件样本：{e}")
    
    for enc in encodings:
        try:
            print(f"尝试使用编码：{enc}")
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"使用编码 {enc} 成功读取文件")
            print(f"数据形状：{df.shape}")
            print(f"列名：{list(df.columns)}")
            return df
        except UnicodeDecodeError as e:
            print(f"编码 {enc} 失败：{str(e)[:100]}")
            continue
        except Exception as e:
            print(f"编码 {enc} 读取失败（非解码错误）：{str(e)[:100]}")
            continue
    
    # 如果所有编码都失败，尝试使用 errors='ignore' 或 'replace'
    print("尝试使用 errors='ignore' 方式读取...")
    for enc in ["utf-8", "gbk", "ansi"]:
        try:
            df = pd.read_csv(path, encoding=enc, encoding_errors='ignore', low_memory=False)
            print(f"使用编码 {enc} 和 errors='ignore' 成功读取文件")
            print(f"数据形状：{df.shape}")
            return df
        except Exception:
            continue
    
    # 最后尝试使用二进制模式读取
    print("尝试使用二进制模式读取...")
    try:
        df = pd.read_csv(path, encoding='latin1', low_memory=False)
        print("使用 latin1 编码成功读取文件")
        return df
    except Exception as e:
        raise ValueError(f"无法解析文件编码：{path}，已尝试所有编码方式。错误：{e}")

# ------------------------------
# 单行坐标转换
# ------------------------------
def convert_coord(row, coord_type: CoordSystem, col_map: dict):
    lng = row[col_map["lng"]]
    lat = row[col_map["lat"]]
    
    if pd.isna(lng) or pd.isna(lat):
        return None, None
    
    try:
        if coord_type == CoordSystem.AMAP:
            wgs_lat, wgs_lng = eviltransform.gcj2wgs(lat, lng)
        else:
            wgs_lat, wgs_lng = eviltransform.bd2wgs(lat, lng)
        
        return wgs_lng, wgs_lat
    except Exception as e:
        print(f"坐标转换失败: ({lat}, {lng}), 类型: {coord_type}, 错误: {e}")
        return None, None

# ------------------------------
# 主流程
# ------------------------------
def process():
    print("=" * 50)
    print("开始覆盖更新 WGS84 坐标")
    print("=" * 50)
    
    try:
        df = load_file(config.input_file)
        print("文件加载成功")
        
        for coord_type, col_map in config.columns_map.items():
            print(f"\n处理 {coord_type.value} 坐标系统...")
            
            required_cols = [
                col_map["lng"],
                col_map["lat"],
                col_map["out_lng"],
                col_map["out_lat"],
            ]
            
            # 检查列是否存在
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"警告：文件中缺少列：{missing_cols}")
                print(f"现有列：{list(df.columns)}")
                
                # 尝试寻找相似的列名
                for missing_col in missing_cols:
                    similar_cols = [col for col in df.columns if missing_col[:4] in col or missing_col[-4:] in col]
                    if similar_cols:
                        print(f"  可能相似的列：{similar_cols}")
                
                # 如果缺少输入列，跳过这个坐标系统
                if col_map["lng"] not in df.columns or col_map["lat"] not in df.columns:
                    print(f"跳过 {coord_type.value}，因为缺少输入列")
                    continue
            
            # 确保输出列存在
            for out_col in [col_map["out_lng"], col_map["out_lat"]]:
                if out_col not in df.columns:
                    print(f"添加缺失的输出列：{out_col}")
                    df[out_col] = None
            
            # 应用转换
            df[[col_map["out_lng"], col_map["out_lat"]]] = df.apply(
                lambda row: pd.Series(
                    convert_coord(row, coord_type, col_map)
                ),
                axis=1
            )
            
            print(f"已更新 {coord_type.value} → WGS84")
            print(f"成功转换数量：{df[col_map['out_lng']].notna().sum()}")
        
        # 保存文件
        df.to_csv(config.output_file, index=False, encoding="utf-8-sig")
        print(f"\n文件已保存：{config.output_file}")
        print(f"保存成功，总行数：{len(df)}")
        
    except Exception as e:
        print(f"处理过程中发生错误：{e}")
        import traceback
        traceback.print_exc()

# ------------------------------
if __name__ == "__main__":
    process()

开始覆盖更新 WGS84 坐标
尝试加载文件：D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_place_result.csv
文件大小：18082270 字节
文件前500字节（十六进制）：636f6d70616e7969642c454e544e414d452cb5c7bcc7bbfab9d82cbeadd3aab7b6cea72cd7a2b2e1b5d8d6b72ccbf9d4dad0d0d5fec7f8bbae2ccaa1b7dd2cb5d8c7f82cb0d9b6c8beadb6c82cb0d9b6c8ceb3b6c82cb8dfb5c2beadb6c82cb8dfb5c2ce...
文件前500字节（文本尝试解码为utf-8）：companyid,ENTNAME,Ǽǻ,ӪΧ,עַ,,ʡ,,ٶȾ,ٶγ,ߵ¾,ߵ
尝试使用编码：utf-8-sig
编码 utf-8-sig 失败：'utf-8' codec can't decode byte 0xb5 in position 18: invalid start byte
尝试使用编码：utf-8
编码 utf-8 失败：'utf-8' codec can't decode byte 0xb5 in position 0: invalid start byte
尝试使用编码：gbk
编码 gbk 失败：'gbk' codec can't decode byte 0xaa in position 164295: illegal multibyte sequence
尝试使用编码：gb2312
编码 gb2312 失败：'gb2312' codec can't decode byte 0xef in position 18420: illegal multibyte sequence
尝试使用编码：gb18030
使用编码 gb18030 成功读取文件
数据形状：(19778, 55)
列名：['companyid', 'ENTNAME', '登记机关', '经营范围', '注册地址', '所在行政区划', '省份', '地区', '百度经度', '百度纬度', '高

In [ ]:

#1. 对两套经纬度数据差异在500m以下的，认为其定位均准确，此处选取百度API获得的经纬度数据
#2. 对于两套经纬度数据差异在500m以上的，认为定位有误，利用openstreetmap进行逆向编码爬取，获得各自经纬度下的对应地址
#3. 将对应地址名称与医院名称进行模糊匹配，模糊匹配分数在80分以上的认为定位正确，选取定位正确的API对应获得的经纬度数据

import pandas as pd
import math
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import requests
from fuzzywuzzy import process
import re
import os
import pickle


d:\swx\Anaconda\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:

import pandas as pd
import numpy as np
import os

# ------------------------------
# 文件读取函数
# ------------------------------
def robust_read_csv(file_path):
    """
    尝试多种编码读取CSV文件
    """
    # 常见的编码列表（按优先级排序）
    encodings = [
        'utf-8-sig',  # 带BOM的UTF-8
        'utf-8',      # 标准UTF-8
        'gbk',        # 简体中文Windows
        'gb2312',     # 简体中文
        'gb18030',    # 中文国家标准
        'latin1',     # ISO-8859-1
        'iso-8859-1', # 另一种拉丁编码
        'cp1252',     # Windows拉丁编码
    ]
    
    for encoding in encodings:
        try:
            print(f"尝试使用 {encoding} 编码读取...")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✓ 使用 {encoding} 编码读取成功，共 {len(df)} 行，{len(df.columns)} 列")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"使用 {encoding} 编码时出错: {str(e)}")
            continue
    
    # 如果所有编码都失败，使用带错误处理的读取
    print("所有编码尝试失败，使用错误忽略模式读取...")
    return pd.read_csv(file_path, encoding='utf-8', errors='ignore')

# ------------------------------
# 数据加载
# ------------------------------
file_path = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_place_baidu_gaode_wgs84.csv"

if os.path.exists(file_path):
    print(f"文件存在，大小: {os.path.getsize(file_path)} 字节")
    df_merged = robust_read_csv(file_path)
else:
    print("错误：文件不存在！")
    exit()
print(f"数据加载完成，共 {len(df_merged)} 条记录")
print("列名:", df_merged.columns.tolist())
print("\n前几行数据:")
print(df_merged.head())

# ------------------------------
# Haversine 距离函数
# ------------------------------
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    计算两个 WGS84 坐标点之间的地面距离（米）
    """
    # 任一坐标缺失 → 返回 NaN
    if pd.isna(lon1) or pd.isna(lat1) or pd.isna(lon2) or pd.isna(lat2):
        return np.nan

    # 转弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371000  # 地球平均半径（米）

    return c * r


# ------------------------------
# 距离计算（百度 vs 高德）
# ------------------------------
df_merged["distance"] = df_merged.apply(
    lambda row: haversine_distance(
        row["百度_wgs84经度1"], row["百度_wgs84纬度1"],
        row["高德_wgs84经度"], row["高德_wgs84纬度"],
    ),
    axis=1
)

# ------------------------------
# 距离统计
# ------------------------------
distance_stats = df_merged["distance"].describe()
print("百度 vs 高德 WGS84 距离偏差统计：")
print(distance_stats)


# ------------------------------
# 基础字段类型规范（防止后续 groupby / 导出问题）
# ------------------------------
for col in ["ENTNAME", "省份"]:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype(str)


# ------------------------------
# 大偏差筛选
# ------------------------------
THRESHOLD = 3_000_000  # 3000 km，理论上异常值
large_deviation = df_merged[df_merged["distance"] > THRESHOLD]

print(f"偏差超过 {THRESHOLD} 米的记录有 {len(large_deviation)} 条")


文件存在，大小: 21689056 字节
尝试使用 utf-8-sig 编码读取...
✓ 使用 utf-8-sig 编码读取成功，共 19778 行，57 列
数据加载完成，共 19778 条记录
列名: ['companyid', 'ENTNAME', '登记机关', '经营范围', '注册地址', '所在行政区划', '省份', '地区', '百度经度', '百度纬度', '高德经度', '高德纬度', '百度_wgs84经度', '百度_wgs84纬度', '高德_wgs84经度', '高德_wgs84纬度', 'distance', 'wgs_lon_true', 'wgs_lat_true', '_merge', '统一社会信用代码', '注册号', '组织机构代码', '法人姓名', '法人称谓', '公司类型编码', '成立日期', '核准日期', '企业状态编码', '注册资金（万元）', '注册资金原始', '注册资金币种', '实收注册资金', '营业期限开始日期', '营业期限终止日期', '行业分类代码', '注销日期', '吊销日期', 'created', '修改时间', '工商纬度', '工商经度', 'province_check_gd', 'province_check_bd', 'true_source', 'gd_address', 'bd_address', 'gd_amenity', 'bd_amenity', 'gd_amenity_clean', 'bd_amenity_clean', 'gd_match_score', 'bd_match_score', '百度经度', '百度纬度', '百度_wgs84经度1', '百度_wgs84纬度1']

前几行数据:
     companyid        ENTNAME                              登记机关  \
0  921057755.0         云南京州医院                            云南省民政厅   
1  910339803.0       凉山州传染病医院  凉山彝族自治州事业单位登记管理局（电话：08343865303）   
2  921089168.0         南昌正骨医院  

In [5]:
from tqdm import tqdm
tqdm.pandas()

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut
import time
import requests
import pandas as pd

# ------------------------------
# 逆地理编码（带重试）
# ------------------------------
def get_address_details_with_retry(lat, lng, retries=3, delay=2):
    for attempt in range(retries):
        try:
            location = geocode(f"{lat}, {lng}", exactly_one=True, language='zh')
            return location.raw.get('address', {}) if location else {}
        except GeocoderTimedOut:
            print(f"第 {attempt + 1} 次尝试超时，{delay} 秒后重试...")
            time.sleep(delay)
        except Exception as e:
            print(f"第 {attempt + 1} 次尝试出现异常: {str(e)}")
            break
    return {}


# ------------------------------
# 初始化地理编码器
# ------------------------------
geolocator = Nominatim(
    user_agent="hospital_geo_validation",
    timeout=30
)

geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)


def get_address_details(lat, lng):
    try:
        location = geocode(f"{lat}, {lng}", exactly_one=True, language='zh')
        return location.raw.get('address', {}) if location else {}
    except Exception as e:
        print(f"逆地理编码失败: {str(e)}")
        return {}


# ------------------------------
# 测试 Nominatim 服务
# ------------------------------
try:
    test_url = "https://nominatim.openstreetmap.org/reverse?lat=39.9&lon=116.4&format=json"
    response = requests.get(test_url, timeout=5)
    print("Nominatim 服务正常" if response.status_code == 200 else "服务返回错误")
except Exception as e:
    print(f"无法连接 Nominatim: {str(e)}")

try:
    location = geolocator.reverse("31.00779569556785,121.40279835486228", language='zh')
    print("测试查询成功:", location.address)
except Exception as e:
    print(f"测试查询失败: {str(e)}")


# ------------------------------
# 省份边界数据（示例）
# ------------------------------
province_bounds = {
    "北京市": {"min_lon": 115.42, "max_lon": 117.51, "min_lat": 39.44, "max_lat": 41.06},
    "天津市": {"min_lon": 116.71, "max_lon": 118.07, "min_lat": 38.56, "max_lat": 40.25},
    # 其他省份边界保持原样
}


def check_province(lon, lat, target_province):
    bounds = province_bounds.get(target_province)
    if not bounds or pd.isna(lon) or pd.isna(lat):
        return False
    return (
        bounds['min_lon'] <= lon <= bounds['max_lon']
        and bounds['min_lat'] <= lat <= bounds['max_lat']
    )


# ------------------------------
# 省份一致性检查
# ------------------------------
df_merged['province_check_gd'] = df_merged.apply(
    lambda row: check_province(
        row['高德_wgs84经度'],
        row['高德_wgs84纬度'],
        row['省份']
    ),
    axis=1
)

df_merged['province_check_bd'] = df_merged.apply(
    lambda row: check_province(
        row['百度_wgs84经度1'],
        row['百度_wgs84纬度1'],
        row['省份']
    ),
    axis=1
)

print("✅ 省份边界检查完成")


服务返回错误
测试查询成功: 上海市第五人民医院, 鹤庆路, 上海市, 闵行, 200240, 中国
✅ 省份边界检查完成


In [7]:
import os
import pickle

# 第六部分：偏差数据分析和逆地理编码
THRESHOLD_MIN = 500
THRESHOLD_MAX = 5000

# 筛选不同阈值范围的偏差数据
large_deviation_500_5000 = df_merged[
    (df_merged['distance'] > THRESHOLD_MIN) &
    (df_merged['distance'] <= THRESHOLD_MAX)
].copy()

large_deviation_5000 = df_merged[
    df_merged['distance'] > THRESHOLD_MAX
].copy()

print(f"中等偏差记录(500–5000米): {len(large_deviation_500_5000)}")
print(f"大偏差记录(>5000米): {len(large_deviation_5000)}")


# 逆地理编码缓存路径
BASE_DIR = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API"
GEOCODE_CACHE_FILE = os.path.join(BASE_DIR, "geocode_cache.pkl")


def load_geocode_cache():
    if os.path.exists(GEOCODE_CACHE_FILE):
        try:
            with open(GEOCODE_CACHE_FILE, 'rb') as f:
                cache = pickle.load(f)
                print(f"已加载逆地理编码缓存，包含 {len(cache)} 条记录")
                return cache
        except Exception as e:
            print(f"加载缓存失败: {e}")
    return {}


def save_geocode_cache(cache):
    try:
        with open(GEOCODE_CACHE_FILE, 'wb') as f:
            pickle.dump(cache, f)
        print(f"逆地理编码缓存已保存，包含 {len(cache)} 条记录")
    except Exception as e:
        print(f"保存缓存失败: {e}")


def get_address_details_with_cache(lat, lng, cache):
    cache_key = f"{lat:.6f},{lng:.6f}"

    if cache_key in cache:
        return cache[cache_key]

    address_details = get_address_details(lat, lng)
    cache[cache_key] = address_details

    if len(cache) % 10 == 0:
        save_geocode_cache(cache)

    return address_details


# 加载缓存
geocode_cache = load_geocode_cache()


# 中等偏差记录的逆地理编码
if len(large_deviation_500_5000) > 0:
    print("开始处理中等偏差记录的逆地理编码...")

    large_deviation_500_5000['gd_address'] = large_deviation_500_5000.progress_apply(
        lambda row: get_address_details_with_cache(
            row['高德_wgs84纬度'],
            row['高德_wgs84经度'],
            geocode_cache
        ),
        axis=1
    )

    large_deviation_500_5000['bd_address'] = large_deviation_500_5000.progress_apply(
        lambda row: get_address_details_with_cache(
            row['百度_wgs84纬度1'],
            row['百度_wgs84经度1'],
            geocode_cache
        ),
        axis=1
    )


# 大偏差记录的逆地理编码
if len(large_deviation_5000) > 0:
    print("开始处理大偏差记录的逆地理编码...")

    large_deviation_5000['gd_address'] = large_deviation_5000.progress_apply(
        lambda row: get_address_details_with_cache(
            row['高德_wgs84纬度'],
            row['高德_wgs84经度'],
            geocode_cache
        ),
        axis=1
    )

    large_deviation_5000['bd_address'] = large_deviation_5000.progress_apply(
        lambda row: get_address_details_with_cache(
            row['百度_wgs84纬度1'],
            row['百度_wgs84经度1'],
            geocode_cache
        ),
        axis=1
    )


# 保存缓存
save_geocode_cache(geocode_cache)


large_deviation_500_5000.to_csv(
    os.path.join(BASE_DIR, "large_deviation_500_5000.csv"),
    index=False,
    encoding="utf-8-sig"
)

large_deviation_5000.to_csv(
    os.path.join(BASE_DIR, "large_deviation_5000.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("逆地理编码完成")


中等偏差记录(500–5000米): 7345
大偏差记录(>5000米): 9840
已加载逆地理编码缓存，包含 17720 条记录
开始处理中等偏差记录的逆地理编码...


100%|██████████| 7345/7345 [00:00<00:00, 128396.88it/s]


开始处理大偏差记录的逆地理编码...


 23%|██▎       | 2264/9840 [00:12<00:57, 131.26it/s] 

逆地理编码缓存已保存，包含 17730 条记录
逆地理编码缓存已保存，包含 17740 条记录


 24%|██▎       | 2321/9840 [00:32<04:00, 31.20it/s] 

... (579 outputs left out)

100%|█████████▉| 9834/9840 [1:00:36<00:03,  1.64it/s]

逆地理编码缓存已保存，包含 20680 条记录


100%|██████████| 9840/9840 [1:00:42<00:00,  2.70it/s]


逆地理编码缓存已保存，包含 20685 条记录
逆地理编码完成


2、百度地点检索API运行后，和高德坐标进行比对、模糊匹配，加入坐标中。

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import chardet
from rapidfuzz import process


# ------------------------------
# 参数设置
# ------------------------------
BASE_DIR = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API"

THRESHOLD_ACCURATE = 500      # ≤500m 认为一致
THRESHOLD_GEOCODE = 500       # >500m 进入逆地理编码
FUZZY_SCORE_THRESHOLD = 80    # 模糊匹配阈值

GEOCODE_CACHE_FILE = os.path.join(BASE_DIR, "geocode_cache.pkl")


# ------------------------------
# 缓存工具
# ------------------------------
def load_geocode_cache():
    if os.path.exists(GEOCODE_CACHE_FILE):
        try:
            with open(GEOCODE_CACHE_FILE, "rb") as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}


def save_geocode_cache(cache):
    with open(GEOCODE_CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)


def get_address_details_with_cache(lat, lng, cache):
    key = f"{lat:.6f},{lng:.6f}"
    if key in cache:
        return cache[key]

    addr = get_address_details(lat, lng)
    cache[key] = addr

    if len(cache) % 10 == 0:
        save_geocode_cache(cache)

    return addr


# ------------------------------
# 规则 #1：500m 内直接选百度
# ------------------------------
df_merged["wgs_lon_true"] = np.where(
    df_merged["distance"] <= THRESHOLD_ACCURATE,
    df_merged["百度_wgs84经度1"],
    np.nan
)

df_merged["wgs_lat_true"] = np.where(
    df_merged["distance"] <= THRESHOLD_ACCURATE,
    df_merged["百度_wgs84纬度1"],
    np.nan
)

df_merged["true_source"] = np.where(
    df_merged["distance"] <= THRESHOLD_ACCURATE,
    "baidu",
    ""
)


# ------------------------------
# 规则 #2：>500m 的进入逆地理编码
# ------------------------------
large_deviation = df_merged[df_merged["distance"] > THRESHOLD_GEOCODE].copy()

geocode_cache = load_geocode_cache()

large_deviation["gd_address"] = large_deviation.progress_apply(
    lambda row: get_address_details_with_cache(
        row["高德_wgs84纬度"],
        row["高德_wgs84经度"],
        geocode_cache
    ),
    axis=1
)

large_deviation["bd_address"] = large_deviation.progress_apply(
    lambda row: get_address_details_with_cache(
        row["百度_wgs84纬度1"],
        row["百度_wgs84经度1"],
        geocode_cache
    ),
    axis=1
)

save_geocode_cache(geocode_cache)


# ------------------------------
# 提取 amenity
# ------------------------------
def extract_amenity(addr):
    return addr.get("amenity", "") if isinstance(addr, dict) else ""


large_deviation["gd_amenity"] = large_deviation["gd_address"].apply(extract_amenity)
large_deviation["bd_amenity"] = large_deviation["bd_address"].apply(extract_amenity)

large_deviation["gd_amenity_clean"] = np.where(
    large_deviation["gd_amenity"].str.contains("医院", na=False),
    large_deviation["gd_amenity"],
    ""
)

large_deviation["bd_amenity_clean"] = np.where(
    large_deviation["bd_amenity"].str.contains("医院", na=False),
    large_deviation["bd_amenity"],
    ""
)


# ------------------------------
# 模糊匹配得分
# ------------------------------
def fuzzy_score(item, name):
    if not item or not name:
        return 0
    match = process.extractOne(item, [name])
    return match[1] if match else 0


large_deviation["gd_match_score"] = [
    fuzzy_score(a, n)
    for a, n in zip(large_deviation["gd_amenity_clean"], large_deviation["ENTNAME"])
]

large_deviation["bd_match_score"] = [
    fuzzy_score(a, n)
    for a, n in zip(large_deviation["bd_amenity_clean"], large_deviation["ENTNAME"])
]


# ------------------------------
# 规则 #3：模糊匹配 ≥ 80 的坐标被认为可信
# ------------------------------
def choose_coord(row):
    gd_ok = row["gd_match_score"] >= FUZZY_SCORE_THRESHOLD
    bd_ok = row["bd_match_score"] >= FUZZY_SCORE_THRESHOLD

    if gd_ok and (row["gd_match_score"] >= row["bd_match_score"]):
        return row["高德_wgs84经度"], row["高德_wgs84纬度"], "gaode"
    if bd_ok:
        return row["百度_wgs84经度1"], row["百度_wgs84纬度1"], "baidu"

    return np.nan, np.nan, ""


chosen = large_deviation.apply(choose_coord, axis=1, result_type="expand")
chosen.columns = ["wgs_lon_true", "wgs_lat_true", "true_source"]

large_deviation[chosen.columns] = chosen


# ------------------------------
# 回写到主数据
# ------------------------------
df_merged.update(
    large_deviation[["wgs_lon_true", "wgs_lat_true", "true_source"]]
)


# ------------------------------
# 保存结果
# ------------------------------
df_merged.to_csv(
    os.path.join(BASE_DIR, "hospital_wgs84_place_result.csv"),
    index=False,
    encoding="utf-8-sig"
)

large_deviation.to_csv(
    os.path.join(BASE_DIR, "large_deviation_with_match_place.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("坐标筛选与匹配流程完成")


100%|██████████| 17185/17185 [00:00<00:00, 127104.83it/s]


坐标筛选与匹配流程完成


In [2]:
import pandas as pd

# 文件路径
file_final = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_wgs84_final_registered.csv"
file_result = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_wgs84_place_result.csv"

# 读取
df_final = pd.read_csv(file_final, encoding="gb18030")
df_result = pd.read_csv(file_result, encoding="utf-8-sig")

# 去除医院名称空格
df_final['ENTNAME'] = df_final['ENTNAME'].str.strip()
df_result['ENTNAME'] = df_result['ENTNAME'].str.strip()

# 统计原始已有坐标数量
before = df_final['wgs_lon_true'].notna().sum()

# 只保留匹配列
df_match = df_result[['ENTNAME','wgs_lon_true','wgs_lat_true']].copy()

# 重命名避免冲突
df_match = df_match.rename(columns={
    'wgs_lon_true':'lon_new',
    'wgs_lat_true':'lat_new'
})

# merge
df = df_final.merge(df_match, on="ENTNAME", how="left")

# 填补空值
df['wgs_lon_true'] = df['wgs_lon_true'].fillna(df['lon_new'])
df['wgs_lat_true'] = df['wgs_lat_true'].fillna(df['lat_new'])

# 删除临时列
df = df.drop(columns=['lon_new','lat_new'])

# 统计填补后坐标数量
after = df['wgs_lon_true'].notna().sum()

# 输出统计
print("原本已有坐标:", before)
print("填补后坐标:", after)
print("新增匹配坐标:", after - before)


# =========================================
# ⭐ 新增：筛选仍然缺失坐标的数据
# =========================================
df_missing = df[
    df['wgs_lon_true'].isna() | df['wgs_lat_true'].isna()
]

print("仍然缺失坐标数量:", len(df_missing))

# 保存缺失坐标文件
missing_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_missing_coordinates.csv"
df_missing.to_csv(missing_file, index=False, encoding="utf-8-sig")

print("缺失坐标文件已保存:", missing_file)


# =========================================
# 保存完整结果
# =========================================
output_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_wgs84_final_registered_filled.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("文件已保存:", output_file)

原本已有坐标: 20898
填补后坐标: 23303
新增匹配坐标: 2405
仍然缺失坐标数量: 15155
缺失坐标文件已保存: D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_missing_coordinates.csv
文件已保存: D:\Hospital_Pharmacy program\hospital\register data\register 2022\coordinates_v3\place_API\hospital_wgs84_final_registered_filled.csv


3、加入工商数据，并转换坐标系（高德-wgs），填补进医院坐标中。

In [3]:
#加入工商数据坐标，设置质量分score
import pandas as pd
import numpy as np

file_path = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_registered_filled.csv"

# 读取
df = pd.read_csv(file_path, encoding="gb18030")

# 创建score列
df["score"] = np.nan

# 条件1：已有WGS84坐标
cond1 = df["wgs_lon_true"].notna() & df["wgs_lat_true"].notna()
df.loc[cond1, "score"] = 10

# 条件2：WGS84坐标为空，但工商坐标存在
cond2 = (
    df["wgs_lon_true"].isna() &
    df["wgs_lat_true"].isna() &
    df["工商经度"].notna() &
    df["工商纬度"].notna()
)


#转换坐标，工商数据——WGS坐标

def gcj2wgs(lng, lat):
    if pd.isna(lng) or pd.isna(lat):
        return None, None
    wgs_lat, wgs_lng = eviltransform.gcj2wgs(lat, lng)
    return wgs_lng, wgs_lat

# ===================== 转换 =====================
# 工商经度 / 工商纬度 → 转成 → 工商_wgs84经度 / 工商_wgs84纬度
df[["工商_wgs84经度", "工商_wgs84纬度"]] = df.apply(
    lambda row: gcj2wgs(row["工商经度"], row["工商纬度"]), axis=1, result_type="expand"
)

# 填补坐标
df.loc[cond2, "wgs_lon_true"] = df.loc[cond2, "工商_wgs84经度"]
df.loc[cond2, "wgs_lat_true"] = df.loc[cond2, "工商_wgs84纬度"]

# 设置score
df.loc[cond2, "score"] = 8

# 保存
output_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_with_score.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

# 统计
print("原始WGS坐标数量:", cond1.sum())
print("工商坐标填补数量:", cond2.sum())
print("最终坐标数量:", df["wgs_lon_true"].notna().sum())
print("文件保存:", output_file)

原始WGS坐标数量: 23303
工商坐标填补数量: 7069
最终坐标数量: 30372
文件保存: D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_with_score.csv


In [1]:
# 加入工商数据坐标，设置质量分score + 筛选缺失坐标生成missing文件
import pandas as pd
import numpy as np
import eviltransform  # 坐标转换库

# 文件路径
file_path = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_registered_filled.csv"

# 读取数据
df = pd.read_csv(file_path, encoding="gb18030")

# 创建score列
df["score"] = np.nan

# 条件1：已有WGS84坐标 → 质量分10
cond1 = df["wgs_lon_true"].notna() & df["wgs_lat_true"].notna()
df.loc[cond1, "score"] = 10

# 条件2：WGS84为空，但工商坐标存在 → 用工商坐标填补，质量分8
cond2 = (
    df["wgs_lon_true"].isna() &
    df["wgs_lat_true"].isna() &
    df["工商经度"].notna() &
    df["工商纬度"].notna()
)

# 坐标转换函数：GCJ-02 → WGS84
def gcj2wgs(lng, lat):
    if pd.isna(lng) or pd.isna(lat):
        return None, None
    wgs_lat, wgs_lng = eviltransform.gcj2wgs(lat, lng)
    return wgs_lng, wgs_lat

# 工商坐标转换为WGS84
df[["工商_wgs84经度", "工商_wgs84纬度"]] = df.apply(
    lambda row: gcj2wgs(row["工商经度"], row["工商纬度"]), axis=1, result_type="expand"
)

# 用转换后的工商坐标填补空的WGS坐标
df.loc[cond2, "wgs_lon_true"] = df.loc[cond2, "工商_wgs84经度"]
df.loc[cond2, "wgs_lat_true"] = df.loc[cond2, "工商_wgs84纬度"]
df.loc[cond2, "score"] = 8

# ===================== 新增：筛选坐标仍然缺失的数据 =====================
missing_coord_df = df[df["wgs_lon_true"].isna() | df["wgs_lat_true"].isna()].copy()

# 保存完整数据（带score+填补后坐标）
output_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_with_score.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

# 保存缺失坐标数据（missing文件）
missing_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_missing_coords.csv"
missing_coord_df.to_csv(missing_file, index=False, encoding="utf-8-sig")

# ===================== 统计输出 =====================
print("="*50)
print("原始已有WGS84坐标数量:", cond1.sum())
print("使用工商坐标填补数量:", cond2.sum())
print("最终有效坐标数量:", df["wgs_lon_true"].notna().sum())
print("坐标仍缺失的数量:", len(missing_coord_df))
print("="*50)
print("完整文件已保存:", output_file)
print("缺失坐标文件已保存:", missing_file)

原始已有WGS84坐标数量: 23303
使用工商坐标填补数量: 7069
最终有效坐标数量: 30372
坐标仍缺失的数量: 8086
完整文件已保存: D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_with_score.csv
缺失坐标文件已保存: D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_missing_coords.csv


## v2 manual coordinate fill after business-coordinate scoring
加入手动查询的坐标：

手动坐标填充：7,420 行
百度：5,475 行
高德：1,945 行
剩余缺失坐标：666 行
Score设为9

In [ ]:
# v2新增：在工商坐标填补和缺失坐标导出之后运行。
# 功能：按医院名称 ENTNAME 匹配手动查询坐标，转换为 WGS84 后另存为 hospital_wgs84_final_all.csv。
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import eviltransform

# 中文列名使用 Unicode 转义，避免不同终端/编辑器造成编码问题。
COL_BD_LON = "\u767e\u5ea6\u7ecf\u5ea6_\u624b\u52a8\u67e5\u8be2"
COL_BD_LAT = "\u767e\u5ea6\u7eac\u5ea6_\u624b\u52a8\u67e5\u8be2"
COL_GD_LON = "\u9ad8\u5fb7\u7ecf\u5ea6_\u624b\u52a8\u67e5\u8be2"
COL_GD_LAT = "\u9ad8\u5fb7\u7eac\u5ea6_\u624b\u52a8\u67e5\u8be2"

base_dir = Path(r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd")
manual_files = [
    base_dir / "handed_coordinates_part5-3601.xlsx",
    base_dir / ("handed_coordinates_part5_" + "\u540e\u534a\u90e8\u5206-11day.xlsx"),
]

# 读取工商坐标填补后的阶段文件；手动坐标填充后的合并结果另存为 final_all 文件。
input_file = base_dir / "hospital_wgs84_final_with_score.csv"
output_file = r"D:\Hospital_Pharmacy program\hospital\register data\register 2022\final data\hospital_registerd\hospital_wgs84_final_all.csv"
output_file = Path(output_file)
missing_file = base_dir / "hospital_wgs84_missing_coords.csv"
backup_file = base_dir / f"hospital_wgs84_final_all_before_overwrite_{datetime.now():%Y%m%d_%H%M%S}.csv"
audit_file = base_dir / "hospital_wgs84_final_all_manual_fill_audit_by_name.csv"


def clean_coord(series):
    """把手动录入的坐标转成数值，并兼容末尾逗号、中文逗号和空单元格。"""
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace("\uff0c", ",", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan}),
        errors="coerce",
    )


def is_valid_lon_lat(lon, lat):
    return lon.notna() & lat.notna() & lon.between(-180, 180) & lat.between(-90, 90)


def bd09_to_wgs84(lon, lat):
    """百度 BD-09 转 WGS84；eviltransform 的输入和返回顺序都是纬度、经度。"""
    if pd.isna(lon) or pd.isna(lat):
        return np.nan, np.nan
    wgs_lat, wgs_lon = eviltransform.bd2wgs(lat, lon)
    return wgs_lon, wgs_lat


def gcj02_to_wgs84(lon, lat):
    """高德 GCJ-02 转 WGS84；eviltransform 的输入和返回顺序都是纬度、经度。"""
    if pd.isna(lon) or pd.isna(lat):
        return np.nan, np.nan
    wgs_lat, wgs_lon = eviltransform.gcj2wgs(lat, lon)
    return wgs_lon, wgs_lat


def read_csv_auto(path):
    for enc in ["utf-8-sig", "gb18030", "utf-8", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not read CSV with expected encodings: {path}")


manual_parts = []
for manual_file in manual_files:
    sheets = pd.read_excel(manual_file, sheet_name=None)
    for sheet_name, df_sheet in sheets.items():
        df_sheet = df_sheet.copy()
        df_sheet["_manual_file"] = manual_file.name
        df_sheet["_manual_sheet"] = sheet_name
        df_sheet["_manual_row"] = np.arange(2, len(df_sheet) + 2)
        manual_parts.append(df_sheet)

manual = pd.concat(manual_parts, ignore_index=True)
manual["ENTNAME"] = manual["ENTNAME"].astype(str).str.strip()

for col in [COL_BD_LON, COL_BD_LAT, COL_GD_LON, COL_GD_LAT]:
    manual[col] = clean_coord(manual[col])

has_baidu = is_valid_lon_lat(manual[COL_BD_LON], manual[COL_BD_LAT])
has_gaode = is_valid_lon_lat(manual[COL_GD_LON], manual[COL_GD_LAT])

manual[["baidu_manual_wgs_lon", "baidu_manual_wgs_lat"]] = manual.apply(
    lambda row: bd09_to_wgs84(row[COL_BD_LON], row[COL_BD_LAT]),
    axis=1,
    result_type="expand",
)
manual[["gaode_manual_wgs_lon", "gaode_manual_wgs_lat"]] = manual.apply(
    lambda row: gcj02_to_wgs84(row[COL_GD_LON], row[COL_GD_LAT]),
    axis=1,
    result_type="expand",
)

# 坐标选择优先级：先用百度手动坐标；百度缺失时用高德手动坐标；两者都缺失则保留空值。
manual["manual_source"] = np.select(
    [has_baidu, has_gaode],
    ["baidu_manual", "gaode_manual"],
    default=pd.NA,
)
manual["manual_wgs_lon"] = np.select(
    [has_baidu, has_gaode],
    [manual["baidu_manual_wgs_lon"], manual["gaode_manual_wgs_lon"]],
    default=np.nan,
)
manual["manual_wgs_lat"] = np.select(
    [has_baidu, has_gaode],
    [manual["baidu_manual_wgs_lat"], manual["gaode_manual_wgs_lat"]],
    default=np.nan,
)
manual["_source_priority"] = np.select([has_baidu, has_gaode], [0, 1], default=2)

manual_lookup = (
    manual.sort_values(["ENTNAME", "_source_priority", "_manual_file", "_manual_row"])
    .drop_duplicates(subset=["ENTNAME"], keep="first")
    .reset_index(drop=True)
)

hospital, hospital_encoding = read_csv_auto(input_file)
hospital["ENTNAME"] = hospital["ENTNAME"].astype(str).str.strip()
backed_up = output_file.exists()
if backed_up:
    shutil.copy2(output_file, backup_file)

before_missing = (hospital["wgs_lon_true"].isna() | hospital["wgs_lat_true"].isna()).sum()

manual_lookup_for_merge = manual_lookup[[
    "ENTNAME", "companyid", "manual_source", "manual_wgs_lon", "manual_wgs_lat",
    "_manual_file", "_manual_sheet", "_manual_row",
    COL_BD_LON, COL_BD_LAT, COL_GD_LON, COL_GD_LAT,
]].rename(columns={"ENTNAME": "manual_ENTNAME", "companyid": "manual_companyid"})

merged = hospital.merge(
    manual_lookup_for_merge,
    left_on="ENTNAME",
    right_on="manual_ENTNAME",
    how="left",
)

fill_mask = (
    (merged["wgs_lon_true"].isna() | merged["wgs_lat_true"].isna())
    & merged["manual_wgs_lon"].notna()
    & merged["manual_wgs_lat"].notna()
)

merged.loc[fill_mask, "wgs_lon_true"] = merged.loc[fill_mask, "manual_wgs_lon"]
merged.loc[fill_mask, "wgs_lat_true"] = merged.loc[fill_mask, "manual_wgs_lat"]

# 仅当 score 为空时给手动填充坐标标记分数；不覆盖已有 score。
if "score" in merged.columns:
    merged.loc[fill_mask & merged["score"].isna(), "score"] = 9

after_missing = (merged["wgs_lon_true"].isna() | merged["wgs_lat_true"].isna()).sum()

audit = merged.loc[fill_mask, [
    "companyid", "ENTNAME", "manual_companyid", "manual_ENTNAME", "manual_source",
    "manual_wgs_lon", "manual_wgs_lat",
    COL_BD_LON, COL_BD_LAT, COL_GD_LON, COL_GD_LAT,
    "_manual_file", "_manual_sheet", "_manual_row",
]].copy()

hospital_output = merged[hospital.columns].copy()
hospital_output.to_csv(output_file, index=False, encoding="utf-8-sig")

missing_coord_df = hospital_output[hospital_output["wgs_lon_true"].isna() | hospital_output["wgs_lat_true"].isna()].copy()
missing_coord_df.to_csv(missing_file, index=False, encoding="utf-8-sig")
audit.to_csv(audit_file, index=False, encoding="utf-8-sig")

print("manual coordinate fill finished")
print(f"hospital file encoding used: {hospital_encoding}")
print(f"backup file if output existed: {backup_file if backed_up else 'no previous output file'}")
print(f"input hospital file: {input_file}")
print(f"output hospital file: {output_file}")
print(f"refreshed missing file: {missing_file}")
print(f"audit file: {audit_file}")
print(f"manual unique hospitals: {len(manual_lookup)}")
print(f"manual hospitals using Baidu: {(manual_lookup['manual_source'] == 'baidu_manual').sum()}")
print(f"manual hospitals using Gaode: {(manual_lookup['manual_source'] == 'gaode_manual').sum()}")
print(f"missing rows before fill: {before_missing}")
print(f"rows filled by ENTNAME: {fill_mask.sum()}")
print(f"missing rows after fill: {after_missing}")
print("filled source counts:")
print(audit["manual_source"].value_counts(dropna=False))


剩余没有坐标的医院，以百度坐标为准（666条），score设为6

In [2]:
import pandas as pd
from pathlib import Path

file_path = Path(r"D:\Hospital_Pharmacy program\data\hospital\coordinates\hospital_wgs84_final_all.csv")

try:
    df = pd.read_csv(file_path, encoding="utf-8-sig")
    out_encoding = "utf-8-sig"
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding="gb18030")
    out_encoding = "gb18030"

lon_col = "wgs_lon_true"
lat_col = "wgs_lat_true"

baidu_lon_col = "百度_wgs84经度"
baidu_lat_col = "百度_wgs84纬度"

gaode_lon_col = "高德_wgs84经度"
gaode_lat_col = "高德_wgs84纬度"

score_col = "scoret" if "scoret" in df.columns else ("score" if "score" in df.columns else "scoret")
if score_col not in df.columns:
    df[score_col] = pd.NA

# 空字符串视为空缺
for col in [lon_col, lat_col, baidu_lon_col, baidu_lat_col, gaode_lon_col, gaode_lat_col]:
    df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 转为数值，便于判断 0
for col in [lon_col, lat_col, baidu_lon_col, baidu_lat_col, gaode_lon_col, gaode_lat_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# wgs 坐标需要填补：经度或纬度任一为空，或任一为 0
wgs_need_fill = (
    df[lon_col].isna() | df[lat_col].isna() |
    (df[lon_col] == 0) | (df[lat_col] == 0)
)

# 百度坐标可用：经纬度均非空且非 0
baidu_available = (
    df[baidu_lon_col].notna() & df[baidu_lat_col].notna() &
    (df[baidu_lon_col] != 0) & (df[baidu_lat_col] != 0)
)

use_baidu = wgs_need_fill & baidu_available

df.loc[use_baidu, lon_col] = df.loc[use_baidu, baidu_lon_col]
df.loc[use_baidu, lat_col] = df.loc[use_baidu, baidu_lat_col]
df.loc[use_baidu, score_col] = 6

# 百度无法填补后，再用高德
wgs_need_fill_after_baidu = (
    df[lon_col].isna() | df[lat_col].isna() |
    (df[lon_col] == 0) | (df[lat_col] == 0)
)

gaode_available = (
    df[gaode_lon_col].notna() & df[gaode_lat_col].notna() &
    (df[gaode_lon_col] != 0) & (df[gaode_lat_col] != 0)
)

use_gaode = wgs_need_fill_after_baidu & gaode_available

df.loc[use_gaode, lon_col] = df.loc[use_gaode, gaode_lon_col]
df.loc[use_gaode, lat_col] = df.loc[use_gaode, gaode_lat_col]
df.loc[use_gaode, score_col] = 5

still_need_fill = (
    df[lon_col].isna() | df[lat_col].isna() |
    (df[lon_col] == 0) | (df[lat_col] == 0)
)

df.to_csv(file_path, index=False, encoding=out_encoding)

print(f"文件已保存：{file_path}")
print(f"score列使用：{score_col}")
print(f"原始 wgs 坐标为空或为0的记录数：{wgs_need_fill.sum()}")
print(f"使用百度_wgs84坐标补全数，score=6：{use_baidu.sum()}")
print(f"使用高德_wgs84坐标补全数，score=5：{use_gaode.sum()}")
print(f"补全后仍为空或为0的记录数：{still_need_fill.sum()}")

文件已保存：D:\Hospital_Pharmacy program\data\hospital\coordinates\hospital_wgs84_final_all.csv
score列使用：score
原始 wgs 坐标为空或为0的记录数：682
使用百度_wgs84坐标补全数，score=6：682
使用高德_wgs84坐标补全数，score=5：0
补全后仍为空或为0的记录数：0
